A script for some figures for cosyne
- make a heatmap of activity aligned across homings
- plot escapes
- plot homings

In [5]:
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar, # (unmatched number of neurons and cluster ids) # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_14may, JAL8_flip4_10may]


session_names = ["JAL4_3rdSept","JAL4_19thSept","JAL4_28aug","JAL4_11thSept",
    "JAL5_8thSept","JAL5_21stSept",
    "JAL6_28mar", "JAL6_flip4_21mar", "JAL6_flip3_18mar", "JAL6_flip5_25mar", 
    "JAL7_sesh8_9apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_sesh9_16apr", "JAL7_23apr",
    "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip3_7may",  "JAL8_14may", "JAL8_flip4_10may"] 

In [6]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.process.process import Process
from JR_test_scripts.escape.escape_utils import load_homing
from JR_test_scripts.FigureSaver import Figure_Saver

import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import matplotlib.patches as patches
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 30
matplotlib.rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded in PDF

%matplotlib inline

In [11]:
"""Load the data"""
c_names = ['shelter_only', 'barrier', 'flipped_barrier']
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
explore_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
homie_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
save_path = make_directory("Z:/Jasmine_Laurence/summary_plots/Cosyne/behavior/")
tuning_data = '_50bins' # '' or '_50bins' or '_25bins
cond_colors = ['#228B22','#FF8C00','#008B8B']
homing_color = '#6A0DAD'
escape_color = '#E63946'

In [ ]:
"""Load data and Extract the homing and escape x and y positions"""
long_homings = True
for e, exp in enumerate(experiments_objects[7:8]):
    session_name = session_names[7]
    # session_name = session_names[e]
    X, Y, h_cond, cond, x_pos, y_pos, bar, barflip, h_e_bool, full_e_bool = load_data(exp)
    h_start, homie_lengths, long_homie_bool, h_cond, e_bool = process_homings(h_e_bool, full_e_bool, h_cond, X, Y)
    if long_homings:
        save_name = session_name + '_Long_homing_escape'
    else:
        long_homie_bool = np.full(long_homie_bool.shape, True)
        save_name = session_name + '_All_homing_escape'
    make_homie_escape_plot(X, Y, h_start, homie_lengths, long_homie_bool, h_cond, e_bool, save_name, save_path)

In [1]:
def load_data(exp):
        # load session
    session = Process(exp).load_session()
    base_path = os.path.join(session.base_path, session.processed_path)

    # full x and y pos
    video_df = pl.read_csv(os.path.join(base_path, "full_video_dataframe.csv"))
    y_pos = video_df["mouse_y_position"].to_numpy()
    x_pos = video_df["mouse_x_position"].to_numpy()
    bar = video_df["barrier_present"].to_numpy()
    barflip = video_df["barrier_flipped"].to_numpy()

    # load homings
    h_on, h_off, homing_bool = load_homing(session, int(np.amax(np.unique(video_df['frames'].to_numpy()))))

    # booleans for homing+escape and explore
    # TODO this is a hack into the escapes to remove the 1s of pause before they start running
    escape = video_df['EscapePeriod'].to_numpy()
    escape_bool = np.full_like(escape, False)
    start = np.where(np.diff(escape.astype(int)) == 1)[0]
    end = np.where(np.diff(escape.astype(int)) == -1)[0]
    for s, e in zip(start, end):
        escape_bool[s+40:e] = True

    h_e_bool = (homing_bool | escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
    full_e_bool = (escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
    cond = np.zeros(len(bar))
    cond[bar] += 1
    cond[barflip] += 1

    X = x_pos[h_e_bool]
    Y = y_pos[h_e_bool]
    h_cond = cond[h_e_bool]

    return X, Y, h_cond, cond, x_pos, y_pos, bar, barflip, h_e_bool, full_e_bool

In [2]:
def process_homings(h_e_bool, full_e_bool, h_cond, X, Y):
    """Find the homing starts and ends
    Subselect complete homings"""

    # starts of homings & escapes
    starts = np.where(np.diff(h_e_bool.astype(int)) > 0)[0] + 1
    ends = np.where(np.diff(h_e_bool.astype(int)) < 0)[0] + 1
    homie_lengths = ends - starts
    counter, h_start, e_start = 0, np.full(len(homie_lengths), 0), np.full(len(homie_lengths)+1, False)
    for i, h in enumerate(homie_lengths):
        if full_e_bool[starts[i]]:
            e_start[i] = True # a bool that tells us which one of the h_starts are escapes
        h_start[i] = counter 
        counter += h
    h_start = np.append(h_start, len(h_cond)) # a list of the start indices of homings + escapes in the homing/escape time period
    e_bool = full_e_bool[h_e_bool] # a boolean that tells us which periods of the homing/escape are escapes

    # where did the mouse start and end
    y_start = Y[h_start[:-1]]
    y_end = Y[h_start[1:]-1]
    long_homie_bool = np.full(len(X), False)
    homie_id = np.full(len(X), 0) # a vector that increases with each homing
    for h, (s,e) in enumerate(zip(y_start, y_end)):
        homie_id[h_start[h]:h_start[h+1]] = h
        if (s < 512) & (e > 700):
            long_homie_bool[h_start[h]:h_start[h+1]] = True

    return h_start, homie_lengths, long_homie_bool, h_cond, e_bool

In [35]:
def make_homie_escape_plot(X, Y, h_start, homie_lengths, long_homie_bool, h_cond, e_bool, save_name, save_path):
    """Make a plot of all the homings and one of all the escapes in this session + condition"""

    fig, axs = plt.subplots(2, 3, figsize=(15, 10))  # Changed to 2 rows, 3 columns with adjusted figsize
    
    # First row is for homing, second row is for escape
    for c in range(3):  # Loop through conditions
        for i in range(len(h_start)-1):
            start = h_start[i]
            length = homie_lengths[i]
            if long_homie_bool[start] & (h_cond[start] == c):
                # Extract the XY positions for this trial
                x_trial = X[start:start+length]
                y_trial = Y[start:start+length]

                # Set limits and plot XY trajectory - now first index is row (0=homing, 1=escape)
                if e_bool[start]:
                    axs[1, c].plot(x_trial, y_trial, color=escape_color, linewidth=1)
                else:
                    axs[0, c].plot(x_trial, y_trial, color=homing_color, alpha = .5, linewidth=1)

    # Set up all subplots with arena features
    for i in [0, 1]:  # rows - behavior type
        for c in range(3):  # columns - condition
            # Add center circle
            circle = plt.Circle((512, 512), 460, color='k', fill=False, linewidth=2)
            axs[i, c].add_patch(circle)
            if c == 1:
                axs[i, c].plot([512-280, 512+460], [512, 512], 'k', linewidth=2)
            elif c == 2:
                axs[i, c].plot([512-460, 512+280], [512, 512], 'k', linewidth = 2)

            # Add red transparent square
            square = patches.Rectangle((437, 886), 150, 90, facecolor='r', alpha=0.4, edgecolor=None, linewidth=0)
            axs[i, c].add_patch(square)

            # Remove ticks and labels for a clean look
            axs[i, c].set_xlim(0, 1024)
            axs[i, c].set_ylim(0, 1024)
            axs[i, c].set_xticks([])
            axs[i, c].set_yticks([])
            axs[i, c].set_frame_on(False)
            axs[i, c].set_aspect('equal')
            axs[i, c].invert_yaxis()

    # Set column titles (conditions)
    for c in range(3):
        axs[0, c].set_title(c_names[c], color = cond_colors[c])
        
    # Set row labels
    axs[0, 0].set_ylabel('Homing')
    axs[1, 0].set_ylabel('Escape')

    # Initialize the Figure_Saver
    figure_saver = Figure_Saver(dir_save=save_path, format_save=["pdf"], overwrite=True)
    figure_saver.save(fig, name_file=save_name)
    figure_saver = Figure_Saver(dir_save=save_path, format_save=["png"], overwrite=True)
    figure_saver.save(fig, name_file=save_name)
    plt.close()

In [29]:
def make_homie_escape_plot_combined(X, Y, h_start, homie_lengths, long_homie_bool, h_cond, e_bool, session_name, save_path):
    """Make a plot of all the homings and one of all the escapes in this session + condition"""

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))  # Changed to 2 rows, 3 columns with adjusted figsize
    
    # First row is for homing, second row is for escape
    for c in range(3):  # Loop through conditions
        for i in range(len(h_start)-1):
            start = h_start[i]
            length = homie_lengths[i]
            if long_homie_bool[start] & (h_cond[start] == c):
                # Extract the XY positions for this trial
                x_trial = X[start:start+length]
                y_trial = Y[start:start+length]

                # Set limits and plot XY trajectory - now first index is row (0=homing, 1=escape)
                if e_bool[start]:
                    axs[c].plot(x_trial, y_trial, color=escape_color, linewidth = 2)
                else:
                    axs[c].plot(x_trial, y_trial, color=homing_color, alpha = .3)

    # Set up all subplots with arena features
    for c in range(3):  # columns - condition
        # Add center circle
        circle = plt.Circle((512, 512), 460, color='k', fill=False, linewidth=2)
        axs[c].add_patch(circle)
        if c == 1:
            axs[c].plot([512-280, 512+460], [512, 512], 'k', linewidth=2)
        elif c == 2:
            axs[c].plot([512-460, 512+280], [512, 512], 'k', linewidth = 2)

        # Add red transparent square
        square = patches.Rectangle((437, 886), 150, 90, facecolor='r', alpha=0.4, edgecolor=None, linewidth=0)
        axs[c].add_patch(square)

        # Remove ticks and labels for a clean look
        axs[c].set_xlim(0, 1024)
        axs[c].set_ylim(0, 1024)
        axs[c].set_xticks([])
        axs[c].set_yticks([])
        axs[c].set_frame_on(False)
        axs[c].set_aspect('equal')
        axs[c].invert_yaxis()

    # Set column titles (conditions)
    for c in range(3):
        axs[c].set_title(c_names[c], color = cond_colors[c])

    # Initialize the Figure_Saver
    figure_saver = Figure_Saver(dir_save=save_path, format_save=["pdf"], overwrite=True)
    figure_saver.save(fig, name_file=session_name + '_All_homing_escape_combined')
    figure_saver = Figure_Saver(dir_save=save_path, format_save=["png"], overwrite=True)
    figure_saver.save(fig, name_file=session_name + '_All_homing_escape_combined')
    plt.close()